# 05 · Вектор управления

Обучения нет. Для пар снимаем активации среднего слоя на эталонах и на плохих ответах и вычитаем
средние; промпт общий и сокращается в разности:

$$
v = \mu_+ - \mu_-, \qquad h^{(\ell)} \leftarrow h^{(\ell)} + \alpha\, v .
$$

При $\alpha = 1$ прибавляется ровно разность средних. Отрицательное $\alpha$ должно портить
поведение предсказуемым образом — это проверка, что направление найдено.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import data, infer, metrics, report

from contextlib import contextmanager

model, tokenizer = infer.load_model()
dev, extended = list(data.load("dev")), list(data.load("test_extended"))
layers = model.model.language_model.layers
LAYER = len(layers) // 2
print(f"слоёв {len(layers)}, берём {LAYER}")

In [ ]:
def mean_activation(texts):
    """Mean hidden state of LAYER over all positions of all texts."""
    captured = []
    handle = layers[LAYER].register_forward_hook(
        lambda mod, args, out: captured.append((out[0] if isinstance(out, tuple) else out).float().mean(dim=1).squeeze(0).cpu()))
    try:
        with torch.no_grad():
            for text in texts:
                model(**tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device))
    finally:
        handle.remove()
    return torch.stack(captured).mean(dim=0)


def rendered(rows, key):
    return [tokenizer.apply_chat_template(r["prompt"] + r[key], tokenize=False, enable_thinking=False) for r in rows]


import torch

vector = mean_activation(rendered(dev[:32], "chosen")) - mean_activation(rendered(dev[:32], "rejected"))
torch.save({"vector": vector, "layer": LAYER}, report.RUNS / "steering.pt")
print(f"норма разности средних {vector.norm():.2f}")

In [ ]:
@contextmanager
def steered(alpha):
    shift = (alpha * vector).to(model.device)

    def hook(mod, args, out):
        hidden = out[0] if isinstance(out, tuple) else out
        shifted = hidden + shift.to(hidden.dtype)
        return (shifted, *out[1:]) if isinstance(out, tuple) else shifted

    handle = layers[LAYER].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


row = next(r for r in extended if r["category"] == "thesis-intro-blocks")
print("ЗАПРОС:", data.request(row))
for alpha in (-1.0, 0.0, 1.0, 2.0):
    with steered(alpha):
        print("═" * 78, f"α = {alpha:+.1f}")
        print(infer.generate(model, tokenizer, [row["prompt"]], max_new_tokens=300)[0])

Судью под хуком не зовём, он бы тоже оказался под вектором.

In [ ]:
for alpha in (1.0, 1.5):
    with steered(alpha):
        report.evaluate(model, tokenizer, f"steer{alpha:.1f}", note=f"steering vector, alpha {alpha}", with_judge=False)
report.show()